## Домашняя работа
ФИО:

## Задание 1
Реализуйте метакласс ThreadSafeSingleton, который обеспечивает создание только одного экземпляра класса, даже в многопоточной среде.
Используйте `from threading import Lock`


### проверка задания 1

In [10]:
from threading import Lock


class ThreadSafeSingleton(type):
    """Метакласс, который создаёт только один экземпляр класса (singleton)
       и делает это безопасно для многопоточности с помощью Lock.
    """
    _instances = {}      # здесь храним созданные экземпляры по классу
    _lock = Lock()       # общий замок для всех singleton-классов

    def __call__(cls, *args, **kwargs):
        # Реализуем отложенное создание (lazy) + защиту от гонок
        if cls not in cls._instances:
            # Если для этого класса ещё нет экземпляра,
            # блокируем создание, чтобы параллельные потоки
            # не создали несколько объектов сразу
            with cls._lock:
                # Двойная проверка — на случай, если другой поток уже успел создать
                if cls not in cls._instances:
                    instance = super().__call__(*args, **kwargs)
                    cls._instances[cls] = instance
        return cls._instances[cls]


class DatabasePool(metaclass=ThreadSafeSingleton):
    """Простой пример пула соединений с базой, использующего ThreadSafeSingleton."""

    def __init__(self):
        # для наглядности просто считаем 'соединения' числами
        self._next_conn_id = 1

    def get_connection(self):
        """Возвращает новое 'соединение' (условное значение)."""
        conn_id = self._next_conn_id
        self._next_conn_id += 1
        return conn_id


# ----- Проверка задания 1 -----

pool1 = DatabasePool()
pool2 = DatabasePool()
pool3 = DatabasePool()

assert pool1 is pool2 is pool3

conn1 = pool1.get_connection()
conn2 = pool2.get_connection()
assert conn1 != conn2

print("pool1 is pool2 is pool3:", pool1 is pool2 is pool3)
print("conn1:", conn1, "conn2:", conn2)
print("Все проверки пройдены.")

pool1 is pool2 is pool3: True
conn1: 1 conn2: 2
Все проверки пройдены.


## Задание 2

Создайте метакласс, который считает, сколько раз создавался каждый класс.

Требования:
1. Метакласс должен иметь атрибут _counters
1. При создании экземпляра класса счетчик должен увеличиваться
1. Добавьте метод get_count(), который возвращает количество созданных экземпляров

### проверка задания 2

In [9]:
class CountInstances(type):
    """Метакласс, считающий, сколько экземпляров каждого класса создано."""
    _counters = {}  # ключ = класс, значение = количество его экземпляров

    def __call__(cls, *args, **kwargs):
        # создаём экземпляр обычным способом
        instance = super().__call__(*args, **kwargs)

        # увеличиваем счётчик для этого класса
        if cls not in cls._counters:
            cls._counters[cls] = 0
        cls._counters[cls] += 1

        return instance

    def get_count(cls):
        """Вернуть количество созданных экземпляров данного класса."""
        return cls._counters.get(cls, 0)


# Примеры классов, использующих метакласс CountInstances
class User(metaclass=CountInstances):
    def __init__(self, name):
        self.name = name


class Product(metaclass=CountInstances):
    def __init__(self, title):
        self.title = title


# ----- Проверка задания 2 -----

user1 = User("Alice")
user2 = User("Bob")
product1 = Product("Laptop")

print(User.get_count())     # Должно быть 2
print(Product.get_count())  # Должно быть 1

# Дополнительно можно проверить assert'ами
assert User.get_count() == 2
assert Product.get_count() == 1
print("Проверка пройдена.")

2
1
Проверка пройдена.


## Задание 3

Создайте метакласс, который автоматически добавляет метод describe() в каждый класс.

Требования:
1. Метод describe() должен возвращать строку с именем класса
1. Используйте метакласс для создания классов Car и Book

### проверка задания 3

In [8]:
class DescribeMeta(type):
    """Метакласс, автоматически добавляющий метод describe() в класс."""

    def __new__(mcls, name, bases, namespace):
        if "describe" not in namespace:
            def describe(self):
                return f"Это объект класса {self.__class__.__name__}"
            namespace["describe"] = describe

        return super().__new__(mcls, name, bases, namespace)


# Классы, использующие метакласс DescribeMeta
class Car(metaclass=DescribeMeta):
    def __init__(self, model):
        self.model = model


class Book(metaclass=DescribeMeta):
    def __init__(self, title):
        self.title = title


# ----- Проверка задания 3 -----

car = Car("Toyota")
book = Book("Python для начинающих")

print(car.describe())   # Должно быть "Это объект класса Car"
print(book.describe())  # Должно быть "Это объект класса Book"

assert car.describe() == "Это объект класса Car"
assert book.describe() == "Это объект класса Book"
print("Проверка пройдена.")

Это объект класса Car
Это объект класса Book
Проверка пройдена.


## Задание 4

Создайте метакласс, который проверяет, что у класса есть метод save(). Можно использовать `__new__`

Требования:
1. Если у класса нет метода save(), метакласс должен выдать ошибку
1. Создайте класс User с методом save()
1. Попробуйте создать класс Message без метода save() (должна быть ошибка)

### проверка задания 4

In [7]:
class SaveMeta(type):
    """Метакласс, который проверяет, что в классе есть метод save()."""

    def __new__(mcls, name, bases, namespace):
        if name != "BaseWithoutSave":  
            if "save" not in namespace:
                raise TypeError(f"Класс {name} должен определять метод save()")

        return super().__new__(mcls, name, bases, namespace)


class User(metaclass=SaveMeta):
    def __init__(self, name):
        self.name = name

    def save(self):
        print(f"Сохраняю пользователя {self.name}")

user = User("Alice")
user.save() 

print("Класс User создан успешно, метод save() есть.")

Сохраняю пользователя Alice
Класс User создан успешно, метод save() есть.
